# 13 — Enhanced LassoCV: Ablation Study (v3)

Does adding age, years_at_club, and T-2 history improve predictions?

**Configs:**
1. **Baseline** — v_2 features (from_q_proj + to_q + pre_player + style_distance + pre_minutes)
2. **+ Age** — baseline + player_season_age + age_squared
3. **+ Tenure** — baseline + years_at_club
4. **+ T-2 Player** — baseline + pre_*_t2 + pre_minutes_t2 + has_t2_history
5. **+ T-2 Team** — baseline + from_q_proj_*_t2
6. **Full v3** — all features combined

Both datasets (ALL, Transfers Only). LassoCV with MultiOutputRegressor.

In [ ]:
import pandas as pd
import numpy as np
import glob, os, warnings
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

parent = glob.glob('/Users/jorgepadilla/Documents/Documents*Jorge*MacBook*')[0]
BASE = os.path.join(parent, 'thesis_data')
V3_DATA = f'{BASE}/processed_data/model_dataset/v_3'

QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels'
]

TEAM_QUALITIES = [
    'DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
    'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME'
]

POSITIONS = ['Midfielder', 'Central Defender', 'Full Back', 'Winger', 'Striker']

NULL_THRESHOLD = 0.50  # drop quality if >50% null for position

In [ ]:
# Load datasets
DATASETS = {
    'ALL': pd.read_parquet(f'{V3_DATA}/enhanced_model_df.parquet'),
    'Transfers Only': pd.read_parquet(f'{V3_DATA}/enhanced_model_df_transfers_only.parquet')
}

for name, d in DATASETS.items():
    print(f'{name}: {d.shape}')

In [ ]:
def get_valid_qualities(df_pos, qualities, prefix='pre_'):
    """Return qualities with <50% null rate for this position subset."""
    valid = []
    for q in qualities:
        col = f'{prefix}{q}'
        if col in df_pos.columns and df_pos[col].isnull().mean() <= NULL_THRESHOLD:
            valid.append(q)
    return valid

def compute_style_distance(df, suffix=''):
    """Euclidean distance between from_q_proj and to_q vectors."""
    sq_diffs = []
    for q in TEAM_QUALITIES:
        from_col = f'from_q_proj_{q}{suffix}'
        to_col = f'to_q_{q}'
        if from_col in df.columns and to_col in df.columns:
            sq_diffs.append((df[from_col] - df[to_col]) ** 2)
    return np.sqrt(sum(sq_diffs)) if sq_diffs else pd.Series(0, index=df.index)

def build_feature_configs(valid_pre_q, valid_pre_q_t2, df_pos):
    """Build feature column lists for each ablation config."""
    # Baseline (v_2 features)
    base_from = [f'from_q_proj_{q}' for q in TEAM_QUALITIES]
    base_to   = [f'to_q_{q}' for q in TEAM_QUALITIES]
    base_pre  = [f'pre_{q}' for q in valid_pre_q]
    baseline  = base_from + base_to + base_pre + ['style_distance', 'pre_minutes']
    
    # Age features
    age_feats = ['player_season_age', 'age_squared']
    
    # Tenure
    tenure_feats = ['years_at_club']
    
    # T-2 player
    t2_player = [f'pre_{q}_t2' for q in valid_pre_q_t2] + ['pre_minutes_t2', 'has_t2_history']
    
    # T-2 team
    t2_team = [f'from_q_proj_{q}_t2' for q in TEAM_QUALITIES] + ['style_distance_t2']
    
    configs = {
        'Baseline':    baseline,
        '+ Age':       baseline + age_feats,
        '+ Tenure':    baseline + tenure_feats,
        '+ T2 Player': baseline + t2_player,
        '+ T2 Team':   baseline + t2_team,
        'Full v3':     baseline + age_feats + tenure_feats + t2_player + t2_team,
    }
    
    # Filter to columns that actually exist and have data
    for k, cols in configs.items():
        configs[k] = [c for c in cols if c in df_pos.columns]
    
    return configs

print('Helper functions ready.')

## Training Loop

In [ ]:
results = []

for ds_name, df_full in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name} ({len(df_full):,} rows)')
    print(f'{"="*60}')
    
    # Add style_distance features
    df_full = df_full.copy()
    df_full['style_distance'] = compute_style_distance(df_full)
    df_full['style_distance_t2'] = compute_style_distance(df_full, suffix='_t2')
    
    for pos in POSITIONS:
        df_pos = df_full[df_full['position'] == pos].copy()
        
        # Train/test split
        train = df_pos[df_pos['to_season'] <= 2023]
        test  = df_pos[df_pos['to_season'] == 2024]
        
        # Valid qualities for this position
        valid_pre_q    = get_valid_qualities(df_pos, QUALITIES, 'pre_')
        valid_pre_q_t2 = get_valid_qualities(df_pos, QUALITIES, 'pre_')  # same qualities, just _t2
        valid_post_q   = get_valid_qualities(df_pos, QUALITIES, 'post_')
        
        # Target columns
        target_cols = [f'post_{q}' for q in valid_post_q]
        
        # Build configs
        configs = build_feature_configs(valid_pre_q, valid_pre_q_t2, df_pos)
        
        print(f'\n  {pos}: {len(train)} train, {len(test)} test, {len(target_cols)} targets')
        
        for cfg_name, feature_cols in configs.items():
            # Prepare X, y — drop rows with NaN in features or targets
            use_cols = feature_cols + target_cols
            train_clean = train[use_cols].dropna()
            test_clean  = test[use_cols].dropna()
            
            X_train = train_clean[feature_cols].values
            y_train = train_clean[target_cols].values
            X_test  = test_clean[feature_cols].values
            y_test  = test_clean[target_cols].values
            
            if len(X_train) < 50 or len(X_test) < 10:
                continue
            
            # LassoCV
            model = MultiOutputRegressor(
                LassoCV(cv=5, max_iter=10000, n_jobs=-1),
                n_jobs=-1
            )
            model.fit(X_train, y_train)
            
            r2_train = r2_score(y_train, model.predict(X_train), multioutput='uniform_average')
            r2_test  = r2_score(y_test,  model.predict(X_test),  multioutput='uniform_average')
            
            results.append({
                'dataset': ds_name,
                'position': pos,
                'config': cfg_name,
                'n_features': len(feature_cols),
                'n_train': len(X_train),
                'n_test': len(X_test),
                'r2_train': r2_train,
                'r2_test': r2_test,
            })
            
            print(f'    {cfg_name:15s}  feats={len(feature_cols):3d}  '
                  f'R\u00b2 train={r2_train:.3f}  test={r2_test:.3f}')

res = pd.DataFrame(results)
print(f'\nTotal experiments: {len(res)}')

## Results: Mean R² by Config

In [ ]:
# Pivot: mean R² test by config and dataset
pivot = res.pivot_table(values='r2_test', index='config', columns='dataset', aggfunc='mean')
pivot = pivot.reindex(['Baseline', '+ Age', '+ Tenure', '+ T2 Player', '+ T2 Team', 'Full v3'])

print('Mean R\u00b2 (test) by config:')
print(pivot.round(4).to_string())

# Delta vs baseline
print('\n\u0394 R\u00b2 vs Baseline:')
delta = pivot.subtract(pivot.loc['Baseline'])
print(delta.round(4).to_string())

In [ ]:
# Grouped bar chart
config_order = ['Baseline', '+ Age', '+ Tenure', '+ T2 Player', '+ T2 Team', 'Full v3']
colors = {'ALL': 'steelblue', 'Transfers Only': 'coral'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, ds_name in enumerate(['ALL', 'Transfers Only']):
    ax = axes[ax_idx]
    ds_res = res[res['dataset'] == ds_name]
    
    # Mean R² per config
    means = ds_res.groupby('config')['r2_test'].mean().reindex(config_order)
    
    bars = ax.bar(range(len(config_order)), means.values, color=colors[ds_name], alpha=0.8)
    ax.set_xticks(range(len(config_order)))
    ax.set_xticklabels(config_order, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Mean R\u00b2 (test)')
    ax.set_title(f'{ds_name}')
    
    # Baseline reference line
    baseline_r2 = means['Baseline']
    ax.axhline(y=baseline_r2, color='red', linestyle='--', alpha=0.5, label=f'Baseline={baseline_r2:.3f}')
    
    # Value labels
    for i, (bar, v) in enumerate(zip(bars, means.values)):
        delta_v = v - baseline_r2
        label = f'{v:.3f}'
        if i > 0:
            label += f'\n({"+" if delta_v >= 0 else ""}{delta_v:.4f})'
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, label,
                ha='center', va='bottom', fontsize=7)
    
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(means.values) * 1.15)

plt.suptitle('v3 Ablation: Mean R\u00b2 by Feature Config', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Results: Per Position

In [ ]:
# Heatmap: R² by position x config for each dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, ds_name in enumerate(['ALL', 'Transfers Only']):
    ax = axes[ax_idx]
    ds_res = res[res['dataset'] == ds_name]
    
    piv = ds_res.pivot_table(values='r2_test', index='position', columns='config')
    piv = piv.reindex(columns=config_order)
    piv = piv.reindex(POSITIONS)
    
    im = ax.imshow(piv.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.5)
    
    ax.set_xticks(range(len(config_order)))
    ax.set_xticklabels(config_order, rotation=30, ha='right', fontsize=7)
    ax.set_yticks(range(len(POSITIONS)))
    ax.set_yticklabels(POSITIONS, fontsize=8)
    ax.set_title(f'{ds_name}')
    
    for i in range(len(POSITIONS)):
        for j in range(len(config_order)):
            v = piv.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=7,
                       color='white' if v < 0.15 or v > 0.45 else 'black')

plt.colorbar(im, ax=axes, shrink=0.8, label='R\u00b2')
plt.suptitle('R\u00b2 by Position \u00d7 Config', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Delta R² vs baseline per position
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, ds_name in enumerate(['ALL', 'Transfers Only']):
    ax = axes[ax_idx]
    ds_res = res[res['dataset'] == ds_name]
    
    piv = ds_res.pivot_table(values='r2_test', index='position', columns='config')
    piv = piv.reindex(columns=config_order, index=POSITIONS)
    
    # Compute delta vs baseline
    delta = piv.subtract(piv['Baseline'], axis=0)
    delta = delta.drop(columns='Baseline')
    
    x = np.arange(len(POSITIONS))
    width = 0.15
    delta_configs = delta.columns.tolist()
    cfg_colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
    
    for i, cfg in enumerate(delta_configs):
        offset = (i - len(delta_configs)/2 + 0.5) * width
        vals = delta[cfg].values
        ax.bar(x + offset, vals, width, label=cfg, color=cfg_colors[i], alpha=0.8)
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(POSITIONS, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('\u0394 R\u00b2 vs Baseline')
    ax.set_title(f'{ds_name}')
    ax.legend(fontsize=6, loc='best')

plt.suptitle('\u0394 R\u00b2 vs Baseline by Position', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
# Full results table
print('Full Results (R\u00b2 test):')
full_table = res.pivot_table(
    values='r2_test', 
    index=['dataset', 'position'], 
    columns='config'
).reindex(columns=config_order)
print(full_table.round(3).to_string())

# Best config per position x dataset
print('\n\nBest config per position:')
for ds_name in ['ALL', 'Transfers Only']:
    print(f'\n  {ds_name}:')
    ds_res = res[res['dataset'] == ds_name]
    for pos in POSITIONS:
        pos_res = ds_res[ds_res['position'] == pos]
        if len(pos_res) > 0:
            best = pos_res.loc[pos_res['r2_test'].idxmax()]
            baseline_r2 = pos_res[pos_res['config'] == 'Baseline']['r2_test'].values[0]
            delta = best['r2_test'] - baseline_r2
            print(f'    {pos:20s}  {best["config"]:15s}  R\u00b2={best["r2_test"]:.3f}  \u0394={delta:+.4f}')

# Overall verdict
print('\n\nOverall Mean R\u00b2 (test):')
overall = res.groupby(['dataset', 'config'])['r2_test'].mean().unstack('config')
overall = overall.reindex(columns=config_order)
print(overall.round(4).to_string())